# BioHub Cell Tracking During Development
## Chapter 47 — Division-Aware Track Birth and Splitting (V4)

**Fix:** V4 uses only the three V12 files already proven to exist in `46DataV12`:

- `chapter34_top200_detections_multisample.csv`
- `chapter36_filtered_detections_multisample.csv`
- `chapter37_refined_track_nodes_multisample.csv`

No `zarr`, no Internet, no Chapter 37 edges/summary files required.

In [1]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import warnings, zipfile
import numpy as np
import pandas as pd

pd.set_option("display.max_columns",250)
pd.set_option("display.width",220)
print("Chapter 47 V4 ready.")

Chapter 47 V4 ready.


In [2]:
@dataclass(frozen=True)
class Config:
    input_root: Path = Path("/kaggle/input")
    output_dir: Path = Path("/kaggle/working/chapter47")
    z_scale: float = 1.625
    y_scale: float = 0.40625
    x_scale: float = 0.40625
    min_parent_age_frames: int = 5
    candidate_window_before_track_end: int = 12
    max_branch_temporal_gap: int = 2
    max_parent_to_branch_distance: float = 18.0
    min_daughter_separation: float = 2.0
    max_daughter_separation: float = 26.0
    min_branch_persistence_frames: int = 3
    accept_score: float = 0.62
    suppress_nearby_frames: int = 3
    max_divisions_per_sample: int = 25

CONFIG=Config()
CONFIG.output_dir.mkdir(parents=True,exist_ok=True)

## 1. Load only the V12 files that actually exist

In [3]:
REQUIRED={
    "ch34":"chapter34_top200_detections_multisample.csv",
    "ch36":"chapter36_filtered_detections_multisample.csv",
    "ch37":"chapter37_refined_track_nodes_multisample.csv",
}

def find_file(filename):
    m=list(CONFIG.input_root.rglob(filename))
    if m: return m[0]
    for zpath in CONFIG.input_root.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zpath) as zf:
                for member in zf.namelist():
                    if Path(member).name==filename:
                        target=Path("/kaggle/working")/filename
                        with zf.open(member) as src, open(target,"wb") as dst:
                            dst.write(src.read())
                        return target
        except zipfile.BadZipFile:
            pass
    return None

paths={k:find_file(v) for k,v in REQUIRED.items()}
missing=[REQUIRED[k] for k,p in paths.items() if p is None]
if missing:
    raise FileNotFoundError("Missing required file(s): "+", ".join(missing))

for k,p in paths.items():
    print(k,p)

ch34 /kaggle/input/datasets/emailchrismathews/46datav12/chapter34_top200_detections_multisample.csv
ch36 /kaggle/input/datasets/emailchrismathews/46datav12/chapter36_filtered_detections_multisample.csv
ch37 /kaggle/input/datasets/emailchrismathews/46datav12/chapter37_refined_track_nodes_multisample.csv


In [4]:
ch34=pd.read_csv(paths["ch34"])
ch36=pd.read_csv(paths["ch36"])
nodes=pd.read_csv(paths["ch37"])

for df in (ch34,ch36,nodes):
    df["sample_id"]=df["sample_id"].astype(str)

samples=sorted(set(ch34["sample_id"]) & set(ch36["sample_id"]) & set(nodes["sample_id"]))
print("Samples:",samples)
print("Rows:",{"ch34":len(ch34),"ch36":len(ch36),"ch37":len(nodes)})

Samples: ['6bba_09961292', '6bba_48816121', '6bba_afb141ff']
Rows: {'ch34': 60000, 'ch36': 15000, 'ch37': 13892}


## 2. Normalize Chapter 37 tracks

In [5]:
def first_existing(df,names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(names)

TRACK=first_existing(nodes,["track_id","global_track_id","refined_track_id"])
T=first_existing(nodes,["t","time","timepoint"])
Z=first_existing(nodes,["z","z_px"])
Y=first_existing(nodes,["y","y_px"])
X=first_existing(nodes,["x","x_px"])

nodes[T]=nodes[T].astype(int)
nodes["z_physical"]=nodes[Z].astype(float)*CONFIG.z_scale
nodes["y_physical"]=nodes[Y].astype(float)*CONFIG.y_scale
nodes["x_physical"]=nodes[X].astype(float)*CONFIG.x_scale
nodes=nodes.sort_values(["sample_id",TRACK,T]).reset_index(drop=True)
nodes["node_key"]=np.arange(len(nodes))

track_meta=(nodes.groupby(["sample_id",TRACK],as_index=False)
            .agg(start_t=(T,"min"),end_t=(T,"max"),node_count=(T,"size")))
track_meta["duration"]=track_meta["end_t"]-track_meta["start_t"]+1
groups={(str(s),tid):g.sort_values(T).reset_index(drop=True)
        for (s,tid),g in nodes.groupby(["sample_id",TRACK])}

display(track_meta.groupby("sample_id").agg(tracks=(TRACK,"size"),median_duration=("duration","median")))

,tracks,median_duration
sample_id,,
6bba_09961292,408,18.0
6bba_48816121,504,15.0
6bba_afb141ff,435,21.0


## 3. Generate label-free division candidates

In [6]:
def xyz(r):
    return np.array([r["z_physical"],r["y_physical"],r["x_physical"]],float)

def dist(a,b):
    return float(np.linalg.norm(xyz(a)-xyz(b)))

def persist(g,start_t):
    q=g[g[T]>=int(start_t)]
    return 0 if q.empty else int(q[T].max()-q[T].min()+1)

def first_between(g,lo,hi):
    q=g[(g[T]>=lo)&(g[T]<=hi)]
    return None if q.empty else q.sort_values(T).iloc[0]

rows=[]
for s in samples:
    sm=track_meta[track_meta["sample_id"]==s]
    for _,pm in sm.iterrows():
        p=pm[TRACK]
        pg=groups[(s,p)]
        if pm["duration"]<CONFIG.min_parent_age_frames: continue
        start=max(int(pm["start_t"])+CONFIG.min_parent_age_frames-1,
                  int(pm["end_t"])-CONFIG.candidate_window_before_track_end)
        for split_t in range(start,int(pm["end_t"])+1):
            prior=pg[pg[T]<=split_t]
            if prior.empty: continue
            parent=prior.iloc[-1]
            branches=[]
            own=first_between(pg,split_t+1,split_t+CONFIG.max_branch_temporal_gap+1)
            if own is not None:
                branches.append({"kind":"parent_suffix","track_id":p,"seed":own,
                                 "start_t":int(own[T]),"persistence":persist(pg,int(own[T]))})
            starts=sm[(sm[TRACK]!=p)&(sm["start_t"]>=split_t+1)&
                      (sm["start_t"]<=split_t+CONFIG.max_branch_temporal_gap+1)]
            for _,dm in starts.iterrows():
                dg=groups[(s,dm[TRACK])]
                branches.append({"kind":"distinct_track","track_id":dm[TRACK],"seed":dg.iloc[0],
                                 "start_t":int(dg.iloc[0][T]),"persistence":int(dm["duration"])})
            plausible=[]
            for b in branches:
                d=dist(parent,b["seed"])
                if d<=CONFIG.max_parent_to_branch_distance and b["persistence"]>=CONFIG.min_branch_persistence_frames:
                    q=dict(b); q["parent_distance"]=d; q["temporal_gap"]=q["start_t"]-split_t
                    plausible.append(q)
            for i in range(len(plausible)):
                for j in range(i+1,len(plausible)):
                    a,b=plausible[i],plausible[j]
                    sep=dist(a["seed"],b["seed"])
                    if not(CONFIG.min_daughter_separation<=sep<=CONFIG.max_daughter_separation): continue
                    rows.append({
                        "sample_id":s,"parent_track_id":p,"split_t":split_t,
                        "daughter_a_kind":a["kind"],"daughter_a_track_id":a["track_id"],
                        "daughter_a_start_t":a["start_t"],"daughter_a_parent_distance":a["parent_distance"],
                        "daughter_a_persistence":a["persistence"],
                        "daughter_b_kind":b["kind"],"daughter_b_track_id":b["track_id"],
                        "daughter_b_start_t":b["start_t"],"daughter_b_parent_distance":b["parent_distance"],
                        "daughter_b_persistence":b["persistence"],
                        "daughter_separation":sep,"max_temporal_gap":max(a["temporal_gap"],b["temporal_gap"])
                    })

candidates=pd.DataFrame(rows)
print("Candidates:",len(candidates))

Candidates: 5375


## 4. Score and select

In [7]:
if len(candidates):
    c=candidates.copy()
    c["score_distance"]=1-np.clip(((c["daughter_a_parent_distance"]+c["daughter_b_parent_distance"])/2)/CONFIG.max_parent_to_branch_distance,0,1)
    c["score_temporal"]=1-np.clip(c["max_temporal_gap"]/(CONFIG.max_branch_temporal_gap+1),0,1)
    cap=8.0
    c["score_persistence"]=np.clip((np.minimum(c["daughter_a_persistence"],cap)+np.minimum(c["daughter_b_persistence"],cap))/(2*cap),0,1)
    pref=8.0
    c["score_separation"]=1-np.clip(np.abs(c["daughter_separation"]-pref)/max(CONFIG.max_daughter_separation-pref,1),0,1)
    c["score_structure"]=((c["daughter_a_kind"]=="parent_suffix")^(c["daughter_b_kind"]=="parent_suffix")).astype(float)
    c["division_score"]=.28*c["score_distance"]+.17*c["score_temporal"]+.22*c["score_persistence"]+.13*c["score_separation"]+.20*c["score_structure"]
    candidates=c.sort_values("division_score",ascending=False).reset_index(drop=True)

selected_rows=[]; claimed={s:set() for s in samples}; parent_frames={}
for _,r in candidates.iterrows():
    if r["division_score"]<CONFIG.accept_score: continue
    s=str(r["sample_id"]); p=r["parent_track_id"]; t=int(r["split_t"]); key=(s,p)
    if any(abs(t-x)<=CONFIG.suppress_nearby_frames for x in parent_frames.get(key,[])): continue
    branch_ids={r["daughter_a_track_id"],r["daughter_b_track_id"]}-{p}
    if any(x in claimed[s] for x in branch_ids): continue
    if sum(1 for x in selected_rows if x["sample_id"]==s)>=CONFIG.max_divisions_per_sample: continue
    selected_rows.append(r.to_dict()); parent_frames.setdefault(key,[]).append(t); claimed[s].update(branch_ids)

selected=pd.DataFrame(selected_rows)
print("Selected:",len(selected))
if len(selected): display(selected.head(20))

Selected: 75


,sample_id,parent_track_id,split_t,daughter_a_kind,daughter_a_track_id,daughter_a_start_t,daughter_a_parent_distance,daughter_a_persistence,daughter_b_kind,daughter_b_track_id,daughter_b_start_t,daughter_b_parent_distance,daughter_b_persistence,daughter_separation,max_temporal_gap,score_distance,score_temporal,score_persistence,score_separation,score_structure,division_score
0,6bba_48816121,271,61,parent_suffix,271,62,1.861671,11,distinct_track,339,62,1.675012,12,3.447146,1,0.901759,0.666667,1.0000,0.747064,1.0,0.882944
1,6bba_48816121,385,91,parent_suffix,385,92,0.574524,8,distinct_track,485,92,2.071477,8,2.437500,1,0.926500,0.666667,1.0000,0.690972,1.0,0.882580
2,6bba_afb141ff,327,61,parent_suffix,327,62,0.574524,9,distinct_track,338,62,1.861671,11,2.187723,1,0.932328,0.666667,1.0000,0.677096,1.0,0.882408
3,6bba_48816121,6,6,parent_suffix,6,7,3.564829,12,distinct_track,120,7,2.187723,39,5.687500,1,0.840207,0.666667,1.0000,0.871528,1.0,0.881890
4,6bba_09961292,116,52,parent_suffix,116,53,0.908403,11,distinct_track,287,53,1.816805,23,2.333729,1,0.924300,0.666667,1.0000,0.685207,1.0,0.881214
5,6bba_afb141ff,118,7,parent_suffix,118,8,2.031250,9,distinct_track,133,8,3.374566,24,5.202538,1,0.849838,0.666667,1.0000,0.844585,1.0,0.881084
6,6bba_09961292,147,75,parent_suffix,147,76,1.284675,12,distinct_track,308,76,6.358818,8,7.501888,1,0.787681,0.666667,1.0000,0.972327,1.0,0.880286
7,6bba_afb141ff,252,86,parent_suffix,252,87,0.574524,11,distinct_track,403,87,5.202538,13,5.420051,1,0.839526,0.666667,1.0000,0.856670,1.0,0.879768
8,6bba_09961292,176,33,parent_suffix,176,34,0.406250,11,distinct_track,215,34,6.942003,22,7.001186,1,0.795882,0.666667,1.0000,0.944510,1.0,0.878967
9,6bba_48816121,197,34,parent_suffix,197,35,2.031250,9,distinct_track,237,35,5.465535,10,6.989389,1,0.791756,0.666667,1.0000,0.943855,1.0,0.877726


## 5. Build Chapter 47 topology from nodes only

In [8]:
split_nodes=nodes.copy()
split_nodes["original_track_id"]=split_nodes[TRACK]
split_nodes["chapter47_track_id"]=split_nodes[TRACK].astype(str)
split_nodes["chapter47_role"]="unchanged"

lineage=[]; transform=[]
for _,ev in selected.iterrows():
    s=str(ev["sample_id"]); p=ev["parent_track_id"]; t=int(ev["split_t"])
    specs=[(1,ev["daughter_a_kind"],ev["daughter_a_track_id"],int(ev["daughter_a_start_t"])),
           (2,ev["daughter_b_kind"],ev["daughter_b_track_id"],int(ev["daughter_b_start_t"]))]
    for n,kind,source,start_t in specs:
        new_id=f"{s}::P{p}::T{t}::D{n}"
        sm=split_nodes["sample_id"].eq(s)
        mask=(sm & split_nodes["original_track_id"].eq(p) & (split_nodes[T]>t)) if kind=="parent_suffix" else              (sm & split_nodes["original_track_id"].eq(source) & (split_nodes[T]>=start_t))
        moved=int(mask.sum())
        split_nodes.loc[mask,"chapter47_track_id"]=new_id
        split_nodes.loc[mask,"chapter47_role"]=f"daughter_{n}"
        lineage.append({"sample_id":s,"parent_track_id":str(p),"daughter_track_id":new_id,
                        "split_t":t,"branch_no":n,"source_kind":kind,
                        "source_original_track_id":str(source),"division_score":float(ev["division_score"])})
        transform.append({"sample_id":s,"parent_track_id":str(p),"split_t":t,"branch_no":n,
                          "source_kind":kind,"source_original_track_id":str(source),
                          "chapter47_track_id":new_id,"moved_nodes":moved})

lineage_edges=pd.DataFrame(lineage)
transform_log=pd.DataFrame(transform)

continuation=[]
for (s,tid),g in split_nodes.groupby(["sample_id","chapter47_track_id"]):
    g=g.sort_values(T)
    rr=g[["node_key",T]].to_dict("records")
    for a,b in zip(rr[:-1],rr[1:]):
        continuation.append({"sample_id":s,"track_id":tid,
                             "source_node_key":a["node_key"],"target_node_key":b["node_key"],
                             "source_t":a[T],"target_t":b[T],"edge_type":"continuation"})
split_edges=pd.DataFrame(continuation)

print("Lineage edges:",len(lineage_edges))
print("Continuation edges:",len(split_edges))

Lineage edges: 150
Continuation edges: 12473


## 6. Save outputs

In [9]:
summary=pd.DataFrame([{
    "samples":len(samples),
    "chapter37_tracks":int(nodes.groupby(["sample_id",TRACK]).ngroups),
    "label_free_candidates":len(candidates),
    "selected_divisions":len(selected),
    "lineage_edges":len(lineage_edges),
    "moved_nodes":int(transform_log["moved_nodes"].sum()) if len(transform_log) else 0
}])
display(summary.T)

outputs={
    "chapter47_division_candidates.csv":candidates,
    "chapter47_selected_divisions.csv":selected,
    "chapter47_split_track_nodes.csv":split_nodes,
    "chapter47_split_track_edges.csv":split_edges,
    "chapter47_lineage_edges.csv":lineage_edges,
    "chapter47_transform_log.csv":transform_log,
    "chapter47_structural_summary.csv":summary,
}
for name,df in outputs.items():
    df.to_csv(CONFIG.output_dir/name,index=False)
    print(name,len(df))

zip_path=Path("/kaggle/working/chapter47_outputs_v4.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in CONFIG.output_dir.glob("*.csv"):
        zf.write(p,p.name)

print("CHAPTER 47 V4 COMPLETE")
print("Portable bundle:",zip_path)

,0
samples,3
chapter37_tracks,1347
label_free_candidates,5375
selected_divisions,75
lineage_edges,150
moved_nodes,1111


chapter47_division_candidates.csv 5375
chapter47_selected_divisions.csv 75
chapter47_split_track_nodes.csv 13892
chapter47_split_track_edges.csv 12473
chapter47_lineage_edges.csv 150
chapter47_transform_log.csv 150
chapter47_structural_summary.csv 1
CHAPTER 47 V4 COMPLETE
Portable bundle: /kaggle/working/chapter47_outputs_v4.zip
